<a href="https://colab.research.google.com/github/hebelmx/ExxerProAIExplorer/blob/main/src/pyApp/notebooks/ExxerproDataSetLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2
!pip install transformers datasets pinecone-client
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client google-drive-api


In [ ]:
!git clone https://github.com/adithya-s-k/omniparse
!cd omniparse


Cloning into 'omniparse'...
remote: Enumerating objects: 579, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 579 (delta 47), reused 45 (delta 39), pack-reused 511
Receiving objects: 100% (579/579), 543.50 KiB | 10.66 MiB/s, done.
Resolving deltas: 100% (286/286), done.


In [ ]:
%cd omniparse

!cd omniparse
!pip install -e .


/content/omniparse
Obtaining file:///content/omniparse
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.8/601.8 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 12.6 MB/s eta 0:00:00
  Prepari

In [ ]:
!ls

omniparse  sample_data


In [ ]:
!pip install  pinecone-client



ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [ ]:
pip install torch

In [ ]:

import pinecone
import os
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="70cb138a-9e88-4be3-b437-207502d72d8a")
# Create a Pinecone index
index_name = "quickstart"

pc.create_index(
    name="quickstart",
    dimension=768, # Assuming 768 dimensions for BERT embeddings
    metric="euclidean", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)


TypeError: Index.__init__() missing 1 required positional argument: 'host'

In [ ]:
index = pc.Index(index_name)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define the folder path
pdf_folder_path = '/content/drive/Drives'
output_folder_path = '/content/drive/Drives/'



Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer, AutoModel

# Load the tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def tokenize_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()  # Get the mean of the token embeddings


In [ ]:
texts = ["Hello world", "Pinecone is great", "Transformers are powerful"]

for i, text in enumerate(texts):
    vector = tokenize_text(text)
    index.upsert([(f"id-{i}", vector)])


NameError: name 'torch' is not defined

In [ ]:
query_vector = tokenize_text("Hello")

# Query Pinecone
results = index.query([query_vector], top_k=3)
for match in results["matches"]:
    print(f"ID: {match['id']}, Score: {match['score']}")


In [ ]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfFileReader(file)
        text = ""
        for page_num in range(reader.numPages):
            text += reader.getPage(page_num).extract_text()
    return text


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "facebook/llama-3b"  # Example model name, replace with the correct one if available
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

nlp = pipeline("text-generation", model=model, tokenizer=tokenizer)


In [ ]:
def generate_qa_pairs(text, nlp):
    # This function should be adapted to the specifics of your task and the model's capabilities
    # Here, we generate a single question-answer pair for each paragraph
    paragraphs = text.split('\n\n')
    qa_pairs = []
    for para in paragraphs:
        if para.strip():
            generated = nlp(f"Generate a question and answer based on the following context: {para}", max_length=150, num_return_sequences=1)
            qa_pairs.append((para, generated[0]['generated_text']))
    return qa_pairs


In [ ]:
import json

def save_qa_pairs(qa_pairs, output_path):
    with open(output_path, 'w') as file:
        json.dump(qa_pairs, file)

pdf_path = '/content/drive/MyDrive/path/to/your/pdf_file.pdf'
output_path = '/content/drive/MyDrive/path/to/your/output_file.json'

text = extract_text_from_pdf(pdf_path)
qa_pairs = generate_qa_pairs(text, nlp)
save_qa_pairs(qa_pairs, output_path)
